# CONNECT Iceberg REST - Time Series Data Retrieval


This notebook demonstrates zero-copy ecosystem consumption of CONNECT Virtual Tables through an open Iceberg-style flow:

# 1. Setup and Parameters

## 1.1 Setup

In [1]:
# =========================================================
# CONFIGURE STYLE SHEET
# =========================================================
from IPython.display import display, HTML

display(HTML("""
<style>

/* Wider notebook */
.jp-Notebook {
    max-width: 96% !important;
    margin: auto;
}

/* Rounded code cells */
.jp-CodeCell {
    border-radius: 14px;
    overflow: hidden;
    margin-bottom: 12px;
}

/* Bigger code font */
.jp-CodeMirrorEditor {
    font-size: 17px !important;
    line-height: 1.5 !important;
}

/* Markdown titles */
.jp-RenderedHTMLCommon h1 {
    font-size: 2.2em;
    font-weight: 700;
}

.jp-RenderedHTMLCommon h2 {
    font-size: 1.6em;
    color: #ffdd19;
}

/* Hide prompt numbers */
.jp-InputPrompt,
.jp-OutputPrompt {
    display: none;
}

</style>
"""))

In [ ]:
# =========================================================
# STEP 1 — VERIFY PYTHON ENVIRONMENT
# =========================================================

import requests
import pandas as pd
import plotly
import fastavro
import fsspec
import pyarrow

print("Environment ready")


In [84]:
# =========================================================
#  IMPORT LIBRARIES
# =========================================================
# requests  : REST calls to the Iceberg Catalog
# fastavro  : Iceberg manifest reading
# fsspec    : ADLS access through SAS token
# pandas    : dataframe processing
# plotly    : modern interactive visualization
# =========================================================

import os
import requests
import urllib3
import pandas as pd
import plotly.express as px
import fsspec

from fastavro import reader

# Demo mode: hide warnings caused by corporate SSL interception.
# For production, configure the corporate root CA instead of verify=False.
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


## 1.2 Connection Parameters

In [ ]:
# =========================================================
# DEFINE CONNECT ICEBERG PARAMETERS
# =========================================================
# Replace TOKEN with your current CONNECT bearer token.
# Recommendation: set CONNECT_ICEBERG_TOKEN as an environment variable
# instead of storing the token directly inside the notebook.
# =========================================================

ICEBERG_ENDPOINT = ""
WAREHOUSE = ""
TOKEN = os.getenv("CONNECT_ICEBERG_TOKEN", "")
VERIFY_SSL = False  # Demo mode only. Use a corporate CA bundle for production.
HEADERS = {"Authorization": f"Bearer {TOKEN}"}


# 2. CONNECT Iceberg REST

## 2.3 Discover Iceberg Namespaces

In [86]:
# =========================================================
# ICEBERG REST HELPER
# =========================================================
# Small helper to keep the notebook clean and readable.
# =========================================================

def iceberg_get(path: str, params: dict | None = None) -> dict:
    # GET an Iceberg REST resource and return JSON.
    url = f"{ICEBERG_ENDPOINT.rstrip('/')}/{path.lstrip('/')}"
    response = requests.get(
        url,
        headers=HEADERS,
        params=params,
        verify=VERIFY_SSL,
        timeout=60
    )
    response.raise_for_status()
    return response.json()


In [ ]:
# =========================================================
# 05 — DISCOVER ICEBERG CATALOG CONFIGURATION
# =========================================================
# The catalog returns the effective prefix used for namespaces/tables.
# =========================================================

catalog_config = iceberg_get(
    "/v1/config",
    params={"warehouse": WAREHOUSE}
)

prefix = catalog_config["overrides"]["prefix"]

print("Catalog prefix:", prefix)
print("Available endpoints:")
for endpoint in catalog_config["endpoints"]:
    print(" -", endpoint)


## 2.4 Discover Iceberg Tables

In [ ]:
# =========================================================
# DISCOVER NAMESPACES AND TABLES
# =========================================================
# Namespaces are equivalent to logical schemas.
# Tables are Iceberg table objects exposed by CONNECT.
# =========================================================

namespaces_payload = iceberg_get(f"/v1/{prefix}/namespaces")
namespaces = [tuple(ns) for ns in namespaces_payload["namespaces"]]

print("Namespaces:", namespaces)

namespace = namespaces[0][0]

tables_payload = iceberg_get(f"/v1/{prefix}/namespaces/{namespace}/tables")
raw_tables = tables_payload.get("identifiers") or tables_payload.get("tables") or []
tables = [t["name"] if isinstance(t, dict) else t[-1] for t in raw_tables]

print("Selected namespace:", namespace)
print("Tables:", tables)

table_name = "spray_dryer_standard_live" if "spray_dryer_standard_live" in tables else tables[0]
print("Selected table:", table_name)


## 2.5 Retrieve Iceberg Metadata

In [ ]:
# =========================================================
# LOAD ICEBERG TABLE METADATA
# =========================================================
# This returns:
# - Iceberg schema
# - current snapshot
# - manifest list
# - temporary ADLS SAS token for data access
# =========================================================

table_payload = iceberg_get(
    f"/v1/{prefix}/namespaces/{namespace}/tables/{table_name}"
)

metadata = table_payload["metadata"]
config = table_payload["config"]

manifest_list_abfss = metadata["snapshots"][0]["manifest-list"]
record_count = metadata["snapshots"][0]["summary"].get("total-records")

print("Table:", table_name)
print("Records:", f"{int(record_count):,}" if record_count else "unknown")
print("Manifest list:", manifest_list_abfss)


# 3. Read and Visualize CONNECT data

## 3.1 Read ADLS Parquet Data

In [ ]:
# =========================================================
# CREATE ADLS FILESYSTEM FROM VENDED SAS TOKEN
# =========================================================
# The Iceberg endpoint provides temporary scoped access to ADLS.
# This is the key zero-copy federation mechanism.
# =========================================================

sas_key = next(k for k in config if k.startswith("adls.sas-token."))
storage_account = sas_key.split(".")[2]
sas_token = config[sas_key]

fs = fsspec.filesystem(
    "abfss",
    account_name=storage_account,
    sas_token=sas_token
)

print("Storage account:", storage_account)
print("SAS token available:", bool(sas_token))


In [ ]:
# =========================================================
# RESOLVE ICEBERG MANIFESTS TO PARQUET DATA FILES
# =========================================================
# Iceberg metadata points to manifest files.
# Manifest files point to the physical Parquet data files.
# =========================================================

with fs.open(manifest_list_abfss, "rb") as f:
    manifest_records = list(reader(f))

manifest_paths = [record["manifest_path"] for record in manifest_records]
print("Manifest files:", len(manifest_paths))

all_data_file_paths = []
for manifest_path in manifest_paths:
    with fs.open(manifest_path, "rb") as f:
        data_file_records = list(reader(f))
    all_data_file_paths.extend(
        record["data_file"]["file_path"]
        for record in data_file_records
    )

print("Parquet data files:", len(all_data_file_paths))
print(all_data_file_paths[0])


## 3.2 Prepare Time Series Model

In [ ]:
# =========================================================
# READ PARQUET DATA INTO PANDAS
# =========================================================
# For the demo table, one Parquet data file contains the full dataset.
# =========================================================

df = pd.concat(
    [pd.read_parquet(path, filesystem=fs) for path in all_data_file_paths],
    ignore_index=True
)

print("Raw shape:", df.shape)
print("Raw columns:", df.columns.tolist())

df.head()


In [ ]:
# =========================================================
# PREPARE ENGINEERING TIME SERIES MODEL
# =========================================================
# Iceberg physical columns are mapped back to logical fields:
#   Timestamp | Name | Value | uom
#
# Then the signal name is split:
#   Dryer02.Motor.Power.PV
#      ↓       ↓     ↓
#    Asset    Unit  Measurement
#
# The fourth field, usually PV, is intentionally discarded.
# =========================================================

# Remap physical Iceberg field IDs to meaningful business names.
df.columns = ["Timestamp", "Name", "Value", "uom"]

# Convert and sort time axis.
df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df = df.sort_values("Timestamp")

# Split signal name into engineering dimensions.
name_parts = df["Name"].str.split(".", expand=True)
df["Asset"] = name_parts[0]
df["Unit"] = name_parts[1]
df["Measurement"] = name_parts[2]

# Keep a clean analytical model.
df = df[["Timestamp", "Asset", "Unit", "Measurement", "Value", "uom"]]

print("Prepared shape:", df.shape)
df.head()


## 3.3 Build Interactive Visualization

In [ ]:
# =========================================================
# FOCUS DEMO WINDOW AND DRYER02 SIGNALS
# =========================================================
# Keep the visualization sharp for a 15–20 second video.
# =========================================================

start_date = "2026-04-24"
end_date = "2026-04-27"

demo_df = df[
    (df["Timestamp"] >= start_date) &
    (df["Timestamp"] <= end_date) &
    (df["Asset"] == "Dryer02") &
    (df["Measurement"].str.lower() != "vibration")
].copy()

demo_df["Signal"] = (
    demo_df["Unit"] + " · " +
    demo_df["Measurement"] + " (" +
    demo_df["uom"].fillna("") + ")"
)

print("Demo shape:", demo_df.shape)
print("Signals:", demo_df["Signal"].unique())


In [ ]:
# =========================================================
# INTERACTIVE TIME SERIES
# =========================================================
# Dark theme, unified hover, faceted by equipment unit.
# Designed for conference video capture.
# =========================================================

fig = px.line(
    demo_df,
    x="Timestamp",
    y="Value",
    color="Signal",
    color_discrete_map=color_map,
    facet_row="Unit",
    title="Dryer02 — Process Signals | CONNECT Iceberg REST → ADLS Parquet",
    hover_data={
        "Timestamp": True,
        "Asset": True,
        "Unit": True,
        "Measurement": True,
        "uom": True,
        "Value": ":.3f",
        "Signal": False
    },
    template="plotly_dark"
)

fig.update_traces(line_width=1.4, mode="lines")

fig.update_layout(
    template="plotly_dark",
    height=650,
    title={
        "text": "Dryer02 — Process Measurements<br><sup>Power, Speed, Temperature and Pressure | Vibration excluded</sup>",
        "x": 0.03,
        "xanchor": "left"
    },
    paper_bgcolor="#1e1e1e",
    plot_bgcolor="#1e1e1e",
    hovermode="x unified",
    legend_title_text="Signal",
    font=dict( family="Segoe UI", size=12 ),
    margin=dict(l=70, r=40, t=100, b=60)
)

fig.update_xaxes(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.08)",
    title_text="Timestamp"
)

fig.update_yaxes(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.08)",
    title_text="Value"
)

color_map = {
    "Fan · Speed (rps)": "#4D69E0",                # Slate Blue
    "Fan · InletTemperature (°C)": "#FF6C1A",     # Bright Orange
    "Fan · InletPressure (bar)": "#30B887",       # Mint Green
    "Motor · Power (kWh)": "#FA9614"              # Golden Yellow
}

fig.show()
